# Online Algorithms Model

## Learning Objectives

1. **Define** the online algorithm model and contrast it with offline computation
2. **Explain** competitive ratio and why it is the right performance measure
3. **Set up** the bipartite matching problem for online advertising
4. **Prove** that greedy achieves competitive ratio 1/2

## Online vs Offline

An **offline** algorithm sees the entire input before making any decisions.
An **online** algorithm must make an irrevocable decision for each input item as it arrives.

| Property | Offline | Online |
|----------|---------|--------|
| Input visibility | Complete | One item at a time |
| Decisions | Can revise | Irrevocable |
| Optimality | Exact optimum possible | Approximate; depends on future |
| Examples | Sorting, shortest path | Ski rental, secretary problem, ad assignment |

**Why online?** In real systems, the future is unknown:
- Search queries arrive one at a time
- Stock prices change moment by moment
- Ad impressions must be assigned instantly

## Competitive Ratio

Let OPT be the value of the optimal offline solution and ALG be the value of the online algorithm.

$$\text{CR}(\text{ALG}) = \min_{\text{inputs}} \frac{\text{ALG}}{\text{OPT}}$$

A CR of $c$ means: on every input, ALG achieves at least $c$ times the offline optimum.

- CR = 1: optimal (as good as offline)
- CR = 1/2: at least half the offline optimum on every input
- CR → 0: terrible — adversary can make ALG arbitrarily bad

**Randomized competitive ratio:** compare to OPT in expectation; adversary cannot see coin flips.

CR is a *worst-case* measure — pessimistic, but gives unconditional guarantees.

## The Bipartite Matching Problem

**Setup:**
- Advertisers $A = \{a_1, \ldots, a_m\}$ known in advance — *offline* side
- Queries $Q = \{q_1, \ldots, q_n\}$ arrive one at a time — *online* side
- Each advertiser can be matched to at most one query
- Edges $E \subseteq A \times Q$: advertiser $a_i$ is eligible for query $q_j$

**Goal:** maximize the number of matched pairs (maximum matching)

**Online constraint:** when query $q_j$ arrives, must assign immediately to an available eligible advertiser (or leave unmatched). Cannot revoke previous assignments.

In [ ]:
from itertools import permutations

def greedy_online_matching(advertisers, query_edges):
    """
    Greedy online matching: assign each query to any available eligible advertiser.
    query_edges: list of sets, query_edges[j] = set of eligible advertisers for query j
    """
    available = set(advertisers)
    matching = {}
    for j, eligible in enumerate(query_edges):
        candidates = eligible & available
        if candidates:
            chosen = next(iter(candidates))   # arbitrary eligible advertiser
            matching[j] = chosen
            available.remove(chosen)
    return matching

def offline_optimal(advertisers, query_edges):
    """Brute-force optimal matching (small instances only)."""""
    best = {}
    for perm in permutations(advertisers):
        matching = {}
        used = set()
        for j, eligible in enumerate(query_edges):
            for a in perm:
                if a in eligible and a not in used:
                    matching[j] = a
                    used.add(a)
                    break
        if len(matching) > len(best):
            best = matching
    return best

# Example where greedy fails
advertisers = ['A', 'B']
# Query 1 can go to A or B; Query 2 can only go to B
query_edges = [{'A','B'}, {'B'}]

greedy = greedy_online_matching(advertisers, query_edges)
optimal = offline_optimal(advertisers, query_edges)
print(f"Greedy matched: {len(greedy)}/2 — {greedy}")
print(f"Optimal:        {len(optimal)}/2 — {optimal}")
print(f"CR on this instance: {len(greedy)/len(optimal):.2f}")

## Why Greedy Achieves CR = 1/2

**Claim:** for any input, greedy matches at least half of what the optimal offline matching achieves.

**Proof sketch:**
Let $M^*$ = optimal matching, $M$ = greedy matching.

For every unmatched query $q$ in $M$: either $q$ has no eligible advertisers, or every eligible advertiser for $q$ was already matched in $M$ (greedy took them first).

Each advertiser matched by greedy "blocks" at most one query from $M^*$ that greedy missed.
Since $|M^*| \leq |M| + |M^* \setminus M|$ and each missed query corresponds to a greedy match:

$$|M| \geq |M^*| - |M| \implies 2|M| \geq |M^*| \implies \text{CR} \geq 1/2$$

**Tight example:** $n$ advertisers and $n+1$ queries; first query eligible for all; remaining queries each eligible for one advertiser. Greedy might use one advertiser on the first query and leave a later query unmatched.

## RANKING Algorithm

The **RANKING** algorithm (Karp, Vazirani, Vazirani 1990) achieves CR = $1 - 1/e \approx 0.632$:

1. Assign each advertiser a uniformly random rank $\sigma(a) \in [0,1]$
2. When query $q$ arrives, assign to the **highest-ranked** available eligible advertiser

This simple change — breaking ties by a pre-committed random ranking — improves CR from 1/2 to $1-1/e$.

**Key insight:** the random ranking prevents the adversary from constructing a worst-case input, because the adversary doesn't know the ranking.

In [ ]:
import random

def ranking_online_matching(advertisers, query_edges, seed=42):
    """RANKING algorithm: pre-assign random ranks; assign to highest-ranked available."""
    rng = random.Random(seed)
    ranks = {a: rng.random() for a in advertisers}
    available = set(advertisers)
    matching = {}
    for j, eligible in enumerate(query_edges):
        candidates = eligible & available
        if candidates:
            chosen = max(candidates, key=lambda a: ranks[a])
            matching[j] = chosen
            available.remove(chosen)
    return matching

# Adversarial example for greedy: RANKING fixes it
import random as rnd
rnd.seed(0)
n = 6
advertisers = list(range(n))
# First query eligible for all; subsequent queries eligible for one advertiser each
query_edges = [set(range(n))] + [{i} for i in range(n-1)]

results = {'greedy': [], 'ranking': [], 'optimal': []}
for trial in range(200):
    g = greedy_online_matching(advertisers, query_edges)
    r = ranking_online_matching(advertisers, query_edges, seed=trial)
    o = offline_optimal(advertisers, query_edges)
    results['greedy'].append(len(g)/max(len(o),1))
    results['ranking'].append(len(r)/max(len(o),1))
    results['optimal'].append(1.0)

print(f"Avg competitive ratio over 200 trials:")
for alg, vals in results.items():
    print(f"  {alg:10s}: {sum(vals)/len(vals):.3f}")
print(f"  Theory (RANKING): {1 - 1/2.718:.3f}")

## Summary

| Algorithm | Competitive Ratio | Notes |
|-----------|------------------|-------|
| Greedy | 1/2 | Simple; deterministic |
| RANKING | 1 − 1/e ≈ 0.632 | Random ranking; optimal for bipartite matching |
| Offline optimal | 1.0 | Sees all queries in advance |

The competitive ratio framework gives rigorous worst-case guarantees for online algorithms.
For the bipartite matching problem:
- 1/2 is achievable with a simple deterministic algorithm
- 1 − 1/e is achievable with randomization
- No randomized algorithm can exceed 1 − 1/e for this problem